In [ ]:
import time
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

logging.getLogger("transformers").setLevel(logging.ERROR)

import sys
sys.path.append("../../utils/")

from utils import *

In [ ]:
# =========================
# CONFIGURACIÓN
# =========================

k = 1
n_folds = 5

dataset_name = "pd_BCCC17__RUS_SMOTE__v1"

batch_size_pred = 16
source_max_token_len = 150
target_max_token_len = 3
use_gpu = False   # CPU

In [ ]:
# =========================
# RUTAS
# =========================

BASE_DIR = Path("../../../").resolve()

ruta_base_dataset = BASE_DIR / "02_datasets" / "processed" / dataset_name

output_path = (
    BASE_DIR
    / "04_experimentos"
    / "modelos"
    / f"{dataset_name}__outputs"
    / f"{dataset_name}__outputs__{k}_{n_folds}"
)

ruta_test = output_path / "test_final"
ruta_test.mkdir(parents=True, exist_ok=True)

In [ ]:
# =========================
# CARGA TEST
# =========================

nombre_test = f"{dataset_name}__test.csv"

df_test = cargar_dataset(nombre_test, ruta_base_dataset)
# o si prefieres:
# df_test = pd.read_csv(ruta_base_dataset / nombre_test)

df_test["source_text"] = df_test["source_text"].astype(str)
df_test["target_text"] = df_test["target_text"].astype(str)

print("Test shape:", df_test.shape)
df_test.head()

In [ ]:
# =========================
# CARGA MODELO FINAL
# =========================

model = SimpleT5Wrapper()
model.from_pretrained("t5", output_path, use_gpu=use_gpu)

print("Modelo cargado desde:", output_path)

In [ ]:
# =========================
# PREDICCIÓN
# =========================

test_texts = df_test["source_text"].tolist()
y_true = df_test["target_text"].tolist()

inicio = time.time()

y_pred = model.predict(
    test_texts,
    batch_size=batch_size_pred,
    source_max_token_len=source_max_token_len,
    target_max_token_len=target_max_token_len
)

fin = time.time()

tiempo_min = (fin - inicio) / 60

print(f"Tiempo de predicción: {tiempo_min:.2f} min")
print("Número de predicciones:", len(y_pred))

In [ ]:
# =========================
# MÉTRICAS GLOBALES
# =========================

accuracy = accuracy_score(y_true, y_pred)

precision_macro = precision_score(y_true, y_pred, average="macro", zero_division=0)
recall_macro = recall_score(y_true, y_pred, average="macro", zero_division=0)
f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)

precision_weighted = precision_score(y_true, y_pred, average="weighted", zero_division=0)
recall_weighted = recall_score(y_true, y_pred, average="weighted", zero_division=0)
f1_weighted = f1_score(y_true, y_pred, average="weighted", zero_division=0)

mcc = matthews_corrcoef(y_true, y_pred)

print("\n=== MÉTRICAS TEST ===")
print(f"Accuracy           : {accuracy:.6f}")
print(f"Precision macro    : {precision_macro:.6f}")
print(f"Recall macro       : {recall_macro:.6f}")
print(f"F1-score macro     : {f1_macro:.6f}")
print(f"Precision weighted : {precision_weighted:.6f}")
print(f"Recall weighted    : {recall_weighted:.6f}")
print(f"F1-score weighted  : {f1_weighted:.6f}")
print(f"MCC                : {mcc:.6f}")
print(f"Tiempo (min)       : {tiempo_min:.2f}")

In [ ]:
# =========================
# GUARDAR MÉTRICAS
# =========================

df_metricas = pd.DataFrame([{
    "fold": k,
    "modelo": output_path.name,
    "accuracy": accuracy,
    "precision_macro": precision_macro,
    "recall_macro": recall_macro,
    "f1_macro": f1_macro,
    "precision_weighted": precision_weighted,
    "recall_weighted": recall_weighted,
    "f1_weighted": f1_weighted,
    "mcc": mcc,
    "tiempo_min": tiempo_min
}])

df_metricas.to_csv(ruta_test / f"metricas_test_fold_{k}.csv", index=False)
df_metricas

In [ ]:
# =========================
# GUARDAR PREDICCIONES
# =========================

df_preds = pd.DataFrame({
    "source_text": test_texts,
    "target_text": y_true,
    "prediction": y_pred
})

df_preds["correct"] = df_preds["target_text"] == df_preds["prediction"]

df_preds.to_csv(ruta_test / f"predicciones_test_fold_{k}.csv", index=False)
df_preds.head()

In [ ]:
# =========================
# CLASSIFICATION REPORT
# =========================

report_dict = classification_report(
    y_true,
    y_pred,
    output_dict=True,
    zero_division=0
)

df_report = pd.DataFrame(report_dict).transpose()
df_report.to_csv(ruta_test / f"classification_report_test_fold_{k}.csv")

df_report

In [ ]:
import re
from collections import Counter

# Clases válidas según la verdad terreno
valid_labels = sorted(set(y_true), key=lambda x: int(x))

# Pasar todo a string limpio
y_pred_str = [str(p).strip() for p in y_pred]

# Predicciones válidas / inválidas
preds_validas = [p for p in y_pred_str if p in valid_labels]
preds_invalidas = [p for p in y_pred_str if p not in valid_labels]

print("=== DIAGNÓSTICO DE PREDICCIONES ===")
print(f"Total predicciones          : {len(y_pred_str)}")
print(f"Predicciones válidas        : {len(preds_validas)}")
print(f"Predicciones inválidas      : {len(preds_invalidas)}")
print(f"Porcentaje inválidas        : {len(preds_invalidas) / len(y_pred_str):.4%}")

# Frecuencias de predicciones inválidas
contador_invalidas = Counter(preds_invalidas)

print("\n=== TOP PREDICCIONES INVÁLIDAS ===")
for pred, n in contador_invalidas.most_common(20):
    print(f"{repr(pred):>20} -> {n}")

# Ver si algunas inválidas contienen números rescatables
def extraer_numeros(texto):
    return re.findall(r"\d+", str(texto))

print("\n=== INVÁLIDAS CON NÚMEROS DENTRO ===")
for pred, n in contador_invalidas.most_common(20):
    nums = extraer_numeros(pred)
    print(f"{repr(pred):>20} -> números: {nums} -> {n}")

# Ejemplos concretos de filas con predicción inválida
df_debug_invalidas = pd.DataFrame({
    "target_text": y_true,
    "prediction_raw": y_pred_str
})

df_debug_invalidas = df_debug_invalidas[
    ~df_debug_invalidas["prediction_raw"].isin(valid_labels)
].copy()

print("\n=== EJEMPLOS DE PREDICCIONES INVÁLIDAS ===")
print(df_debug_invalidas.head(20))

# Guardarlo por si quieres revisarlo fuera
df_debug_invalidas.to_csv(ruta_test / f"predicciones_invalidas_test_fold_{k}.csv", index=False)

print(f"\nCSV de inválidas guardado en: {ruta_test / f'predicciones_invalidas_test_fold_{k}.csv'}")